# 02 · Baselines B1 (PatchCore), B3 (detector + template report), B2 (zero-shot VLMs)
All thresholds are chosen on **val**. Do not run `--split test` here.

## 0. Session setup (run every session)
Code is cloned from GitHub (`moghbaie/InspectEdge_Mobile_VLM`); nothing needs to be uploaded.
Google Drive (`MyDrive/InspectEdge`) only stores outputs: reports, manifests, checkpoints.
Runtime → Change runtime type → **GPU** (T4 is enough for 500M; L4/A100 for 3B).

In [ ]:
import os, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO  = 'https://github.com/moghbaie/InspectEdge_Mobile_VLM.git'
DRIVE = '/content/drive/MyDrive/InspectEdge'   # persistent outputs: reports, manifests, checkpoints
WORK  = '/content/InspectEdge'                 # fast local disk: code + raw images
os.makedirs(DRIVE, exist_ok=True)
if os.path.isdir(f'{WORK}/.git'):
    !git -C {WORK} pull -q --ff-only
else:
    !git clone -q {REPO} {WORK}
assert os.path.exists(f'{WORK}/pyproject.toml'), 'clone failed: check REPO URL / repo visibility'
%cd {WORK}
!git log --oneline -1
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps
os.environ['INSPECTEDGE_ROOT'] = WORK
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, transformers, peft
assert transformers.__version__ == '4.57.1', 'pinned transformers not active: Runtime -> Restart session, then re-run this cell'
print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'

Mounted at /content/drive
/content/InspectEdge
2936c3d (HEAD -> main, origin/main, origin/HEAD) Colab setup: clone code from GitHub instead of Drive; Drive stores outputs only
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing edita

In [ ]:
# Data: official VisA release (CC BY 4.0). Verified by SHA-256 before use.
VISA_SHA = '2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362'
if not os.path.exists('data/raw/visa/pcb1'):
    os.makedirs('data/raw/_dl', exist_ok=True); os.makedirs('data/raw/visa', exist_ok=True)
    !wget -q -O data/raw/_dl/VisA_20220922.tar https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar
    h = subprocess.run(['sha256sum', 'data/raw/_dl/VisA_20220922.tar'], capture_output=True, text=True).stdout.split()[0]
    assert h == VISA_SHA, f'VisA archive checksum mismatch: {h}'
    !tar -xf data/raw/_dl/VisA_20220922.tar -C data/raw/visa && rm data/raw/_dl/VisA_20220922.tar
!ls data/raw/visa

candle	  cashew      fryum	       macaroni1  pcb1	pcb3  pipe_fryum
capsules  chewinggum  LICENSE-DATASET  macaroni2  pcb2	pcb4  split_csv


In [ ]:
# Manifest: rebuilt deterministically each session; fails loudly if the locked split changed.
!python -m inspectedge.data.build_manifest

near-duplicate pairs merged: 15
split lock verified
{
  "train": {
    "n": 7576,
    "sha256_of_ids": "3de0ef677a4806b20b0c2bfbb6a13b3052b053948789293413bb0a1ae485cba1"
  },
  "val": {
    "n": 1623,
    "sha256_of_ids": "9a5735c0bb610a28f712134feec905a12b9a7ac510ac66e3c2877c2644085c3f"
  },
  "test": {
    "n": 1622,
    "sha256_of_ids": "b2267be39ad119a87c155a2dd886fe54ce069c2872c373f37f23cd6deed381d6"
  }
}


## B1 · PatchCore on the official VisA 1-class split (+ corruption suite, severity 3)
~3–5 min/category on T4.

In [ ]:
!python scripts/baseline_patchcore.py --corrupt 2>&1 | grep -v Warning

Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth
100%|██████████| 132M/132M [00:00<00:00, 172MB/s]
candle {"n_train": 900, "n_test": 200, "load_s": 9.3594, "fit_s": 117.4303, "bank_size": 7056, "infer_ms_per_img": 8.1358, "image_auroc": 0.9855, "image_auprc": 0.9853, "image_f1_max": 0.949, "f1_max_threshold": 0.3481, "prevalence": 0.5, "n": 200, "pixel_auroc": 0.9897, "aupro_0.3": 0.9419}
capsules {"n_train": 542, "n_test": 160, "load_s": 12.7644, "fit_s": 42.4513, "bank_size": 4249, "infer_ms_per_img": 5.9222, "image_auroc": 0.6715, "image_auprc": 0.8175, "image_f1_max": 0.7722, "f1_max_threshold": 0.5925, "prevalence": 0.625, "n": 160, "pixel_auroc": 0.9534, "aupro_0.3": 0.6846}
cashew {"n_train": 450, "n_test": 150, "load_s": 7.589, "fit_s": 29.5935, "bank_size": 3528, "infer_ms_per_img": 5.8047, "image_auroc": 0.973, "image_auprc": 0.9876, "image_f1_max": 0.9417, "f1_max_threshold": 0.

In [ ]:
r = json.load(open('reports/baseline_patchcore/metrics.json'))
import pandas as pd
cats = [c for c in r if not c.startswith('_')]
pd.DataFrame({c: {k: r[c][k] for k in ['image_auroc','image_auprc','image_f1_max','pixel_auroc','aupro_0.3']} for c in cats}).T.round(3)

,image_auroc,image_auprc,image_f1_max,pixel_auroc,aupro_0.3
candle,0.986,0.985,0.949,0.990,0.942
capsules,0.671,0.818,0.772,0.953,0.685
cashew,0.973,0.988,0.942,0.990,0.884
chewinggum,0.988,0.995,0.974,0.981,0.842
fryum,0.945,0.975,0.903,0.950,0.758
macaroni1,0.919,0.913,0.851,0.980,0.914
macaroni2,0.649,0.652,0.691,0.927,0.779
pcb1,0.963,0.960,0.919,0.995,0.900
pcb2,0.953,0.949,0.906,0.980,0.873
pcb3,0.944,0.947,0.875,0.981,0.805


In [ ]:
rob = pd.DataFrame({c: {f: v['image_auroc'] for f, v in r[c]['robustness_sev3'].items()} for c in cats}).T
rob.insert(0, 'clean', [r[c]['image_auroc'] for c in cats]); rob.round(3)

,clean,low_light,defocus_blur,motion_blur,sensor_noise,jpeg,glare,combined
candle,0.986,0.626,0.882,0.757,0.680,0.514,0.561,0.510
capsules,0.671,0.714,0.688,0.503,0.437,0.508,0.699,0.532
cashew,0.973,0.436,0.799,0.691,0.504,0.756,0.369,0.557
chewinggum,0.988,0.908,0.975,0.753,0.813,0.730,0.620,0.641
fryum,0.945,0.801,0.682,0.545,0.664,0.583,0.576,0.596
macaroni1,0.919,0.600,0.678,0.582,0.431,0.438,0.503,0.551
macaroni2,0.649,0.575,0.518,0.539,0.453,0.536,0.535,0.540
pcb1,0.963,0.940,0.597,0.516,0.798,0.629,0.700,0.643
pcb2,0.953,0.577,0.612,0.580,0.764,0.474,0.514,0.593
pcb3,0.944,0.665,0.704,0.520,0.685,0.647,0.545,0.575


## B3 · Detector + template report on the VLM val split (freezes per-category thresholds)

In [ ]:
!python scripts/b3_detector_report.py --split val 2>&1 | grep -v Warning | tail -25

macaroni2 done
pcb1 done
pcb2 done
pcb3 done
pcb4 done
pipe_fryum done
{
  "n": 1623,
  "json_parse_rate": 1.0,
  "schema_valid_rate": 1.0,
  "anomaly_balanced_acc": 0.855024255024255,
  "anomaly_recall_flagged": 0.8666666666666667,
  "false_reassurance_rate": 0.13333333333333333,
  "unsupported_finding_rate_on_normals": 0.15661815661815662,
  "abstain_or_invalid_rate": 0.0,
  "status_acc_strict": 0.8459642637091805,
  "defect_type_macro_f1": 0.015594541910331383,
  "defect_type_acc": 0.06666666666666667,
  "region_best_iou_mean": 0.19967714290452088,
  "region_iou>=0.3_rate": 0.29444444444444445,
  "false_reassurance_rate_ci95": [
    0.08554186419408812,
    0.17938664596273288
  ]
}


## B2 · Zero-shot small VLMs (same prompt/schema as fine-tuning)

In [ ]:
for m in ['HuggingFaceTB/SmolVLM-256M-Instruct', 'HuggingFaceTB/SmolVLM-500M-Instruct']:
    !python scripts/eval_vlm.py --model {m} --split val --per-cell 10 2>&1 | grep -v Warning | tail -30

  "unsupported_finding_rate_on_normals": 0.0,
  "abstain_or_invalid_rate": 1.0,
  "status_acc_strict": 0.0,
  "defect_type_macro_f1": 0.0,
  "defect_type_acc": 0.0,
  "region_best_iou_mean": 0.0,
  "region_iou>=0.3_rate": 0.0,
  "false_reassurance_rate_ci95": [
    0.0,
    0.0
  ],
  "unsupported_finding_rate_on_normals_ci95": [
    0.0,
    0.0
  ],
  "schema_valid_rate_ci95": [
    0.0,
    0.0
  ],
  "model": "HuggingFaceTB/SmolVLM-256M-Instruct",
  "adapter": null,
  "split": "val",
  "corrupt": null,
  "severity": null,
  "longest_edge": 1024,
  "latency_p50_s": 1.052219467499981,
  "latency_p95_s": 16.371980547350063,
  "gpu": "Tesla T4",
  "peak_gpu_mem_gb": 1.260859392
}
  "unsupported_finding_rate_on_normals": 0.0,
  "abstain_or_invalid_rate": 1.0,
  "status_acc_strict": 0.0,
  "defect_type_macro_f1": 0.0,
  "defect_type_acc": 0.0,
  "region_best_iou_mean": 0.0,
  "region_iou>=0.3_rate": 0.0,
  "false_reassurance_rate_ci95": [
    0.0,
    0.0
  ],
  "unsupported_finding_rate

Optional quality challenger (needs L4/A100, 4-bit; research comparison only — not deployable on phone):

In [ ]:
# !python scripts/eval_vlm.py --model Qwen/Qwen2.5-VL-3B-Instruct --load-4bit --split val --per-cell 5

In [ ]:
!python scripts/compare_and_gate.py --split val --system B3=reports/b3_detector_report/val_predictions.jsonl --system B2_500M=reports/vlm_eval/SmolVLM-500M-Instruct_val_predictions.jsonl

# Comparison on `val` (240 matched images)

| System | schema_valid_rate | anomaly_recall_flagged | unsupported_finding_rate_on_normals | false_reassurance_rate | defect_type_macro_f1 | region IoU>=0.3 |
|---|---|---|---|---|---|---|
| B3 | 1.000 PASS | 0.850 PASS | 0.192 FAIL | 0.150 PASS | 0.016 | 0.308 |
| B2_500M | 0.000 FAIL | 0.000 FAIL | 0.000 PASS | 0.000 PASS | 0.000 | 0.000 |

## Paired McNemar tests on status correctness

- B3 vs B2_500M: acc 0.829 vs 0.000; discordant 199/0; p=0.0000


In [ ]:
# Persist outputs to Drive (run at the end and after long steps)
!rsync -aR reports data/manifests {DRIVE}/
print('synced to', DRIVE)

synced to /content/drive/MyDrive/InspectEdge
